# AutoARIMA benchmarks

Install sktime

```bash
    uv add sktime
```


In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys
# sys.path.append('.')
# sys.path.append('./DynaMix/')

plt.rcParams["font.family"] = "Helvetica"

In [2]:
from dysts.metrics import smape


def vpt_smape(x, xhat, threshold=30):
    """
    Find the first time index at which an array exceeds a threshold.

    Args:
        x (np.ndarray): The ground truth, a time series of shape (nt, 1).
        xhat (np.ndarray): The forecast, a time series of shape (nt, 1).
        threshold (float): The threshold to search for.

    Returns:
        int: The first time index at which the array exceeds the threshold.
    """
    arr = horizoned_smape(x, xhat)
    exceed_times = np.where(arr > threshold)[0]
    if len(exceed_times) == 0:
        tind = len(arr)
    else:
        tind = exceed_times[0]
    return tind

def horizoned_smape(x, xhat):
    """Given a horizoned forecast and ground truth, compute the SMAPE as a function of time"""
    nt = min(x.shape[0], xhat.shape[0])
    smape_vals = list()
    for i in range(1, nt+1):
        smape_vals.append(smape(x[:i], xhat[:i]))
    smape_vals = np.array(smape_vals)
    return smape_vals


def mse(y_true, y_pred):
    
    mse = np.mean((y_true - y_pred) ** 2)
    return mse

def mse_rolling(ts1, ts2):

    n = min(ts1.shape[0], ts2.shape[0])
    all_mse = list()
    for i in range(1, n+1):
        mse_val = mse(ts1[:i], ts2[:i])
        all_mse.append(mse_val)

    return np.array(all_mse)

def mae(y_true, y_pred):
    
    mae = np.mean(np.abs(y_true - y_pred))
    return mae

def mae_rolling(ts1, ts2):

    n = min(ts1.shape[0], ts2.shape[0])
    all_mae = list()
    for i in range(1, n+1):
        mae_val = mae(ts1[:i], ts2[:i])
        all_mae.append(mae_val)

    return np.array(all_mae)

# Run on real data

Uncomment the particular lines correspond to each data block of interest

In [3]:
from pmdarima import auto_arima

In [ ]:
RE_VAL = 900 # Change this to 300, 600, 900, 1200
import random

import sys
sys.path.append('..')
from models.parrot import context_parroting_forecast
from dysts.metrics import estimate_kl_divergence

SPLIT = "train" # train or test

## Turbulence
# X_pca = np.load(f"../data/von_karman_street/vortex_street_vorticities_Re_{RE_VAL}_pca10.pkl", allow_pickle=True)
# downsamplerate = 2
# data = X_pca[::downsamplerate,:]
# print(data.shape)
# context_length = 512
# forecast_length = 300
# num_ic = 200
# num_modes = 2

# fpath = f"../data/electrocardiogram/ecg_train.csv.gz"
# data = np.loadtxt(fpath, delimiter=",")
# data = data.reshape(-1, 1)
# downsamplerate = 1
# data = data[::downsamplerate,:]
# num_modes = 1
# context_length = 512
# forecast_length = 300
# num_ic = 200
# num_modes = 1



## Electronic Circuit
fpath = "../data/electronic_circuit/R1/ST_1_1.dat"
data = np.loadtxt(fpath)
downsamplerate = 1
data = data[::downsamplerate,:]
context_length = 512
forecast_length = 300
num_ic = 50
num_modes = 28


## Oscillator
# fpath = "../data/kuramoto/oscillator_glass.npy"
# data = np.load(fpath)
# downsamplerate = 1
# data = data[::downsamplerate,:]
# context_length = 512
# forecast_length = 300
# num_ic = 50
# num_modes = 23



evaluated_steps = 50


print(f"Data shape: {data.shape}")


model_mse = []
model_mae = []
model_kl = []
model_mse_rolling = list()
model_mae_rolling = list()



traj = data[:,:num_modes]
# normalize the trajectory
traj = (traj - np.mean(traj, axis=0)) / np.std(traj, axis=0)


for ic in range(num_ic):
    
    start = random.randint(0, data.shape[0] - context_length - forecast_length)
    traj_context = traj[start:start+context_length,:]
    traj_true = traj[start+context_length:start+context_length+forecast_length,:]

    traj_pred = list()
    for mode in range(traj_context.shape[1]):

        traj_pred_mode = auto_arima(traj_context[:, mode], error_action='ignore', trace=False,
                              suppress_warnings=True, maxiter=5,
                              seasonal=True, m=12)
        traj_pred.append(traj_pred_mode.predict(forecast_length))
    traj_pred = np.array(traj_pred).T

    if traj_pred.ndim == 1:
        traj_pred = traj_pred[:, None]

    for mode in range(num_modes):

        mse_val = mse_rolling(traj_pred[:,mode], traj_true[:,mode])
        model_mse_rolling.append(np.array(mse_val))
        model_mse.append(mse_val[evaluated_steps])
        mae_val = mae_rolling(traj_pred[:,mode], traj_true[:,mode])
        model_mae_rolling.append(np.array(mae_val))
        model_mae.append(mae_val[evaluated_steps])

        # best_index, min_l2_distance, Parrot_pred_single = context_parroting_forecast(traj_context[:,mode], D=15, tau=1, forecast_total_length=forecast_length)
        # mse_val = mse_rolling(Parrot_pred_single, traj_true[:,mode])
        # mae_val = mae_rolling(Parrot_pred_single, traj_true[:,mode])

    kl_dist = estimate_kl_divergence(traj_true, traj_pred)
    if np.isinf(kl_dist):
        kl_dist = np.nan
    model_kl.append(kl_dist)



model_average_mae = np.mean(model_mae_rolling, axis=0)
model_average_mse = np.mean(model_mse_rolling, axis=0)

print(f"MSE: {np.mean(model_mse)}, {np.std(model_mse)}")
print(f"MAE: {np.mean(model_mae)}, {np.std(model_mae)}")
print(f"KL: {np.nanmean(model_kl)}, {np.nanstd(model_kl)}")


colors = ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"]

# Plot Average VPT vs. Context Length for each equation
fig = plt.figure()
ax = fig.add_subplot(111)

for axis in ['bottom','left']:
  ax.spines[axis].set_linewidth(5)
for axis in ['top','right']:
  ax.spines[axis].set_linewidth(5)
  #ax.spines[axis].set_visible(False)

ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()

fig.set_size_inches(17,15)
plt.xticks(fontsize = 50)
plt.yticks(fontsize = 50)
plt.xlabel(r'Steps', fontname="Helvetica", fontsize = 60)
plt.ylabel(r'Error', fontname="Helvetica", fontsize = 60)

plt.plot(model_average_mae, lw=5, ls='-', alpha=1, color=colors[1], label='MAE')
plt.plot(model_average_mse, lw=5, ls='-', alpha=1, color=colors[3], label='MSE')

#plt.xscale('log')
#plt.yscale('log')

#plt.xlim([.1, 10])
#plt.ylim([0, 110])

#plt.yticks([0, 50, 100])

plt.legend(loc='upper left', frameon=False, prop={'size':45, 'family': 'Helvetica'}, ncol=1)

plt.gca().tick_params(axis='y', pad=15, size=23, width=5)
plt.gca().tick_params(axis='x', pad=20, size=23, width=5)

fig.set_tight_layout(True)
#plt.savefig(f'mse_rolling_compare.pdf', bbox_inches='tight')


Data shape: (30000, 28)


/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/nump

In [14]:
print(f"MSE: {np.mean(model_mse)}, {np.std(model_mse)}")
print(f"MAE: {np.mean(model_mae)}, {np.std(model_mae)}")
print(f"KL: {np.nanmean(model_kl)}, {np.nanstd(model_kl)}")

MSE: 0.27319718270671717, 0.29229489750476817
MAE: 0.3880164513747167, 0.1788225999918161
KL: 0.15808749575821787, 0.0691797587854401


In [5]:
print(f"MSE: {np.mean(model_mse)}, {np.std(model_mse)}")
print(f"MAE: {np.mean(model_mae)}, {np.std(model_mae)}")
print(f"KL: {np.nanmean(model_kl)}, {np.nanstd(model_kl)}")

MSE: 1.1314892366129383, 0.9962444034990495
MAE: 0.8284493242082018, 0.33542846135650745
KL: 0.3315718299567307, 0.4318100832390417


In [ ]:
# Turbulence
MSE: 0.19933803143734288, 0.23467457686106852
MAE: 0.3154715005987871, 0.16611905494890983
KL: 0.14845963369529028, 0.07543800931084332

# ECG
MSE: 0.5864349504067565, 0.27172708229621706
MAE: 0.5977049578627777, 0.16650337041752408
KL: 0.17521087631042434, 0.12379156498411442

# Circuit
MSE: 0.06351815496387263, 0.041550789575999496
MAE: 0.1950870522146034, 0.06784053642978953
KL: 1.4837784084069812, 0.22820889769876887

# Oscillator
MSE: 3.099189404519514e-05, 0.0001204297328664709
MAE: 0.0032847438595385563, 0.0030354943702306262
KL: 0.5443645722852647, 0.14905436456239599

Dysts dataset

In [4]:
import numpy as np
import matplotlib.pyplot as plt
# from scripts.utils import vpt_smape, vpt_nrmse, vpt_mse, smape_rolling
from scipy.spatial.distance import cdist
from dysts.systems import get_attractor_list
from dysts.flows import __dict__ as flow_models
from dysts.analysis import gp_dim
from dysts.metrics import estimate_kl_divergence
import os

plt.rcParams["font.family"] = "Helvetica"

granularity = 30
#context_lengths = 2**np.arange(6, 14)
context_lengths = 2**np.arange(6, 11)
forecast_length = 300
#simulation_length = 10000
simulation_length = 100000
rolling_window = context_lengths[-1]
num_ic = 20

# Directory to store trajectories
#trajectory_dir = "trajectories"
trajectory_dir = "../data/long_trajectories"


# equation_names = get_attractor_list()
# equation_names.remove('FluidTrampoline')
# equation_names.remove('HyperLu')
# equation_names.remove('SprottMore')
# equation_names.remove('StickSlipOscillator')
#equation_names = equation_names[:2]

# import torch
# import sys
# # sys.path.append('.')
# sys.path.append('./DynaMix/')
# # from src.model.dynamix import DynaMix
# from src.model.forecaster import DynaMixForecaster
# # from src.metrics.metrics import geometrical_misalignment, temporal_misalignment, MASE
# # from src.utilities.plotting_eval import plot_3D_attractor, plot_2D_attractor, plot_TS_forecast
# from src.utilities.utilities import load_hf_model


sys.path.append('../benchmark/')
from metrics import vpt_smape, smape_rolling


# def mse(y_true, y_pred):
    
#     mse = np.mean((y_true - y_pred) ** 2)
#     return mse

# def mse_rolling(ts1, ts2):

#     n = min(ts1.shape[0], ts2.shape[0])
#     all_mse = list()
#     for i in range(1, n+1):
#         mse_val = mse(ts1[:i], ts2[:i])
#         all_mse.append(mse_val)

#     return np.array(all_mse)

# def mae(y_true, y_pred):
    
#     mae = np.mean(np.abs(y_true - y_pred))
#     return mae

# def mae_rolling(ts1, ts2):

#     n = min(ts1.shape[0], ts2.shape[0])
#     all_mae = list()
#     for i in range(1, n+1):
#         mae_val = mae(ts1[:i], ts2[:i])
#         all_mae.append(mae_val)

#     return np.array(all_mae)

In [5]:
average_vpt = dict()
median_vpt = dict()
average_smape = dict()
median_smape = dict()

average_vpt_2 = dict()
median_vpt_2 = dict()
average_smape_2 = dict()
median_smape_2 = dict()

average_cdim = dict()
median_cdim = dict()
average_kl = dict()
median_kl = dict()

average_mse = dict()
median_mse = dict()
average_mae = dict()
median_mae = dict()

In [ ]:
import glob

# average_vpt = np.load(f'../analysis/Moirai_statistics/average_vpt.npy',allow_pickle='TRUE').item()
# found_systems = np.unique([item[0] for item in average_vpt.keys()])

for trajectory in sorted(glob.glob("../data/long_trajectories/*.npy")):
    equation_name = trajectory.split("/")[-1].split(".")[0]
    print(equation_name, flush=True)
    # if equation_name in found_systems:
    #     print(f"Skipping {equation_name} because it's already in the found_systems list", flush=True)
    #     continue
    
    traj = np.load(trajectory, allow_pickle=True)
    traj = (traj - np.mean(traj, axis=0)) / np.std(traj, axis=0)

    try:
        for context_length in context_lengths:

            all_vpt = list() # collecting all vpt for the parroting model
            all_smape_rolling = list() # collecting all smape values across time
            all_vpt_2 = list() # collecting all vpt for the parroting model
            all_smape_rolling_2 = list() # collecting all smape values across time
            all_cdim = list() # collecting all correlation dimensions
            all_kl_dist = list() # collecting all kl divergence values
            all_mse_rolling = list()
            all_mae_rolling = list()

            for i in range(0, len(traj)-context_length-forecast_length, rolling_window):
                if i < rolling_window*num_ic:
                    #print(i//rolling_window)
                    traj_context = traj[i:i+context_length,:]
                    traj_true = traj[i+context_length:i+context_length+forecast_length,:]
                    traj_pred_full = np.zeros_like(traj_true)

                    # Load the pre-trained model
                    # model = load_hf_model("dynamix-3d-alrnn-v1.0")
                    # model.eval() # Set model to evaluation mode
                    # forecaster = DynaMixForecaster(model) # Initialize the forecaster
                    # ## Make DynaMix prediction
                    # context_traj_tensor  = torch.tensor(traj_context)
                    # with torch.no_grad(): 
                    #     reconstruction = forecaster.forecast(
                    #         context=context_traj_tensor,
                    #         horizon=forecast_length, # Match the horizon of the ground truth
                    #         standardize=True,
                    #     )
                    # # reconstruction = torch.transpose(reconstruction, 0, 1)
                    # traj_pred_full = reconstruction.detach().numpy()

                    # traj_pred = list()
                    # for dim_ind in range(traj_context.shape[1]):
                    #     traj_pred_dim = moirai2_forecast(traj_context[:, dim_ind], forecast_length)
                    #     traj_pred.append(traj_pred_dim)
                    # traj_pred_full = np.array(traj_pred).T

                    context_tensor = torch.tensor(traj_context[None, :], dtype=torch.float32)
                    pft_model = PatchTSTPipeline.from_pretrained(
                        mode="predict",
                        pretrain_path="GilpinLab/panda-72M",
                        device_map="cpu",
                    )
                    traj_pred_full = pft_model.predict(
                        context_tensor, forecast_length, limit_prediction_length=False, sliding_context=True,
                    ).squeeze().cpu().numpy()


                    for dim in range(traj_true.shape[1]):
                        traj_pred = traj_pred_full[:, dim]

                        vpt = vpt_smape(traj_pred, traj_true[:, dim]) / granularity
                        smape_val = np.array(smape_rolling(traj_true[:, dim], traj_pred))

                        all_vpt.append(vpt)
                        all_smape_rolling.append(smape_val)

                        mse_val = mse_rolling(traj_true[:, dim], traj_pred)
                        all_mse_rolling.append(mse_val)
                        
                        mae_val = mae_rolling(traj_true[:, dim], traj_pred)
                        all_mae_rolling.append(mae_val)


                    vpt = vpt_smape(traj_pred_full.squeeze(), traj_true.squeeze()) / granularity
                    smape_val = np.array(smape_rolling(traj_true, traj_pred_full))
                    all_vpt_2.append(vpt)
                    all_smape_rolling_2.append(smape_val)
                    
                    kl_dist = estimate_kl_divergence(traj_true, traj_pred_full)
                    if np.isinf(kl_dist):
                        kl_dist = np.nan
                    all_kl_dist.append(kl_dist)

                    cdim_pred = gp_dim(traj_pred_full)
                    cdim_true = gp_dim(traj_true)
                    all_cdim.append(np.array([cdim_pred, cdim_true]))

            average_vpt[(equation_name,context_length)] = np.mean(all_vpt)
            median_vpt[(equation_name,context_length)] = np.median(all_vpt)
            average_smape[(equation_name,context_length)] = np.mean(all_smape_rolling, axis=0)
            median_smape[(equation_name,context_length)] = np.median(all_smape_rolling, axis=0)

            average_vpt_2[(equation_name,context_length)] = np.mean(all_vpt_2)
            median_vpt_2[(equation_name,context_length)] = np.median(all_vpt_2)
            average_smape_2[(equation_name,context_length)] = np.mean(all_smape_rolling_2, axis=0)
            median_smape_2[(equation_name,context_length)] = np.median(all_smape_rolling_2, axis=0)

            average_cdim[(equation_name,context_length)] = np.mean(all_cdim, axis=0)
            median_cdim[(equation_name,context_length)] = np.median(all_cdim, axis=0)
            average_kl[(equation_name,context_length)] = np.nanmean(all_kl_dist)
            median_kl[(equation_name,context_length)] = np.nanmedian(all_kl_dist)

            average_mse[(equation_name,context_length)] = np.mean(all_mse_rolling, axis=0)
            median_mse[(equation_name,context_length)] = np.median(all_mse_rolling, axis=0)
            average_mae[(equation_name,context_length)] = np.mean(all_mae_rolling, axis=0)
            median_mae[(equation_name,context_length)] = np.median(all_mae_rolling, axis=0)

    except Exception as e:
        print(e)
        print(f"Skipping {equation_name}", flush=True)
        continue


Aizawa


/Users/william/program_repos/parroting/benchmark/panda/patchtst/pipeline.py:110: UserWarning: We recommend keeping prediction length <= 128. The quality of longer predictions may degrade since the model is not optimized for it. 
  warnings.warn(msg)


AnishchenkoAstakhov
Arneodo
ArnoldBeltramiChildress
ArnoldWeb


/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: divide by zero encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: overflow encountered in matmul
  return x @ self._LP
/Users/william/program_repos/parroting/.venv/lib/python3.13/site-packages/scipy/stats/_covariance.py:633: RuntimeWarning: invalid value encountered in matmul
  return x @ self._LP


AtmosphericRegime
BeerRNN
BelousovZhabotinsky
BickleyJet
Blasius
BlinkingRotlet
BlinkingVortex
Bouali
Bouali2
BurkeShaw
CaTwoPlus
CaTwoPlusQuasiperiodic
CellCycle
CellularNeuralNetwork
Chen
ChenLee
Chua
CircadianRhythm
CoevolvingPredatorPrey
Colpitts
Coullet
Dadras
DequanLi
DoubleGyre
DoublePendulum
Duffing
ExcitableCell
Finance


In [ ]:
np.save(f'../analysis/panda_statistics/average_vpt.npy', average_vpt)
np.save(f'../analysis/panda_statistics/median_vpt.npy', median_vpt) 
np.save(f'../analysis/panda_statistics/average_smape.npy', average_smape)
np.save(f'../analysis/panda_statistics/median_smape.npy', median_smape)
np.save(f'../analysis/panda_statistics/average_vpt_2.npy', average_vpt_2)
np.save(f'../analysis/panda_statistics/median_vpt_2.npy', median_vpt_2) 
np.save(f'../analysis/panda_statistics/average_smape_2.npy', average_smape_2)
np.save(f'../analysis/panda_statistics/median_smape_2.npy', median_smape_2)
np.save(f'../analysis/panda_statistics/average_cdim.npy', average_cdim)
np.save(f'../analysis/panda_statistics/median_cdim.npy', median_cdim)
np.save(f'../analysis/panda_statistics/average_kl.npy', average_kl)
np.save(f'../analysis/panda_statistics/median_kl.npy', median_kl)
np.save(f'../analysis/panda_statistics/average_mse.npy', average_mse)
np.save(f'../analysis/panda_statistics/median_mse.npy', median_mse)
np.save(f'../analysis/panda_statistics/average_mae.npy', average_mae)
np.save(f'../analysis/panda_statistics/median_mae.npy', median_mae)